# 03 ContinueVariable Inference

## 1. Data

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve().parent
PROCESSED_PATH = ROOT / "data" / "processed" / "yrbs_combined_processed.csv"
TAB_DIR = ROOT / "outputs" / "tables"
SUM_DIR = ROOT / "outputs" / "summary"

BEHAVIOR_VAR = "SadOrHopeless"
BEHAVIOR_P0 = 0.30
CONT_VAR = "HowMuchDoYouWeighWithoutShoesInKG"
CONT_MU0 = 68.0

processed = pd.read_csv(PROCESSED_PATH)
print("Processed data shape:", processed.shape)

Processed data shape: (14041, 10)


## 2.Mean Inference: `HowMuchDoYouWeighWithoutShoesInKG`

In [5]:
w = processed[CONT_VAR].dropna()
n_w = int(w.shape[0])
mean_w = float(w.mean())
sd_w = float(w.std(ddof=1))
se_w = sd_w / np.sqrt(n_w)

t_crit = stats.t.ppf(0.975, df=n_w - 1)
ci_low_w = mean_w - t_crit * se_w
ci_high_w = mean_w + t_crit * se_w

t_stat, p_value_w = stats.ttest_1samp(w, popmean=CONT_MU0)
p_value_display = "< 0.0001" if p_value_w < 0.0001 else f"{p_value_w:.4f}"

mean_results = pd.DataFrame({
    "quantity": [
        "sample_size", "sample_mean", "sample_sd",
        "benchmark_mu0", "t_statistic", "p_value", "ci_low", "ci_high"
    ],
    "value": [
        str(n_w),
        f"{mean_w:.4f}",
        f"{sd_w:.4f}",
        f"{CONT_MU0:.4f}",
        f"{t_stat:.4f}",
        p_value_display,
        f"{ci_low_w:.4f}",
        f"{ci_high_w:.4f}"
    ]
})

decision_w = "reject H0" if p_value_w < 0.05 else "fail to reject H0"

display(mean_results)

#Save
mean_summary = pd.DataFrame({
    "analysis": ["Population mean"],
    "variable": [CONT_VAR],
    "estimate": [f"{mean_w:.4f}"],
    "benchmark": [f"{CONT_MU0:.4f}"],
    "test_statistic": [f"{t_stat:.4f}"],
    "p_value": [
        "< 0.0001" if p_value_w < 0.0001 else f"{p_value_w:.4f}"
    ],
    "ci_low": [f"{ci_low_w:.4f}"],
    "ci_high": [f"{ci_high_w:.4f}"],
    "decision_5pct": [decision_w]
})

mean_summary.to_csv(TAB_DIR / "03_mean_inference_summary.csv", index=False)

,quantity,value
0,sample_size,13062
1,sample_mean,68.5502
2,sample_sd,16.9909
3,benchmark_mu0,68.0000
4,t_statistic,3.7007
5,p_value,0.0002
6,ci_low,68.2588
7,ci_high,68.8416


### Interpretation (Mean Analysis)

- We used the sample to estimate and test whether the population mean of **HowMuchDoYouWeighWithoutShoesInKG** differs from the benchmark of **68.0 kg**.
- The sample mean is **68.5502 kg**, so the observed average weight is only slightly above **68.0 kg** in this sample.
- Because the t-statistic (**3.7007**) is larger than the two-sided critical t value for this sample, we reject the null hypothesis at the 5% significance level.
- Since the benchmark mean of 68.0 kg is not contained in the 95% confidence interval, the null hypothesis is rejected at the 5% significance level.
- Because the **p-value is 0.0002**, we reject the null hypothesis at the 5% significance level and conclude that the population mean is significantly above **68.0 kg**.
- This is consistent with the EDA, where the sample mean was above the benchmark and the distribution showed a mild right tail.
- A cautious point is that the result is statistically significant, but the difference is only about **0.55 kg**, so the practical difference is fairly small.